# Lecture 12 Activity – Who wrote more of the joint essays?

QSS 20 · Modern Statistical Computing · Fall 2026

## The question

Federalist 18, 19, and 20 compare the proposed Constitution with earlier
confederacies: the Greek leagues, the German empire, and the Dutch
provinces. Hamilton and Madison wrote the three together, from notes both
had collected. The lecture settled the 12 disputed essays with two routes:
similarity to each author's essays by subject words, and the rates of style
words such as `upon` and `whilst`.

> **In each of the three joint essays, whose hand shows more, and how far
> can word counts settle that?**

### How to work

Work in pairs for 30 minutes. Submit the nearest essays of each joint essay,
their similarity to each author, their style rates, their topic, and a short
claim for each essay. If you run out of time, finish Task 5 after class.

- The row unit is **one essay**. The setup reads the 85 files and labels the
  authors as the lecture did.
- Use the lecture's settings: `stop_words='english'`, `min_df=2`, four
  topics with `random_state=0`.
- In the starter cells, replace each `...` with code.

In [ ]:
import os
os.environ['OMP_NUM_THREADS'] = '1'

from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.decomposition import NMF
from sklearn.feature_extraction.text import (
    CountVectorizer, TfidfVectorizer)
from sklearn.metrics.pairwise import cosine_similarity

RAW = Path('../../data/raw')

rows = []
for n in range(1, 86):
    path = RAW / 'federalist' / f'fp{n:02d}.txt'
    rows.append({'essay': n,
                 'text': path.read_text(encoding='utf-8')})
essays = pd.DataFrame(rows)

authors = {
    'Hamilton': ([1, 6, 7, 8, 9, 11, 12, 13, 15, 16, 17]
                 + list(range(21, 37)) + [59, 60, 61]
                 + list(range(65, 86))),
    'Madison': [10, 14] + list(range(37, 49)),
    'Jay': [2, 3, 4, 5, 64],
    'Joint': [18, 19, 20],
    'Disputed': list(range(49, 59)) + [62, 63],
}
label = {n: author for author, ns in authors.items() for n in ns}
essays['author'] = essays['essay'].map(label)
counts = CountVectorizer().fit_transform(essays['text'])
essays['n_words'] = np.asarray(counts.sum(axis=1)).ravel()

# Check: 85 essays, three of them joint.
essays['author'].value_counts()

## Task 1 · The nearest essays of each joint essay

Build the tf-idf matrix with the lecture's settings and the 85 by 85 cosine
similarity table, with essay numbers as row and column labels. For each of
essays 18, 19, and 20, list the five nearest essays (not itself) with their
similarity and author.

Do the neighbors come from one author?



<details><summary>Hint</summary>

`TfidfVectorizer(stop_words='english', min_df=2)`, then
`cosine_similarity`. Wrap the result in a DataFrame with
`index=essays['essay'], columns=essays['essay']`. For one essay,
`sim[18].drop(18).sort_values(ascending=False).head(5)`, and `.map(label)`
on the index gives the authors.

</details>

In [ ]:
tfidf_vec = TfidfVectorizer(stop_words='english', min_df=2)
tfidf = tfidf_vec.fit_transform(essays['text'])
terms = tfidf_vec.get_feature_names_out()
sim = pd.DataFrame(cosine_similarity(tfidf),
                   index=..., columns=...)
sim.shape

In [ ]:
for n in [18, 19, 20]:
    nearest = sim[n].drop(n).sort_values(ascending=False).head(...)
    print(pd.DataFrame({'similarity': nearest.round(3),
                        'author': nearest.index.map(label)}), '\n')

*Your answer:*

## Task 2 · Similarity to each author's essays

For every essay, compute the mean similarity to Hamilton's 51 essays and to
Madison's 14, leaving each essay's similarity to itself out. Show the two
numbers for essays 18, 19, and 20, and the difference between them.

Which author is each joint essay closer to, and is the margin large next to
the margins of the 12 disputed essays in the lecture (0.03 to 0.07)?



<details><summary>Hint</summary>

Copy `sim`, set its diagonal to `np.nan` with `np.fill_diagonal`, then
`known[authors['Hamilton']].mean(axis=1)`.

</details>

In [ ]:
known = sim.copy()
np.fill_diagonal(known.values, np.nan)
essays['to_hamilton'] = known[...].mean(axis=1).values
essays['to_madison'] = known[...].mean(axis=1).values
essays['margin'] = essays['to_madison'] - essays['to_hamilton']
joint = essays[essays['author'] == 'Joint']
joint[['essay', 'to_hamilton', 'to_madison', 'margin']].round(3)

*Your answer:*

## Task 3 · The style words

Count `upon`, `whilst`, `while`, and `on` in every essay with a fixed
vocabulary, and turn the counts into rates per 1,000 words. Show the raw
counts and the rates for essays 18, 19, and 20, then the mean rates by
author for comparison.

Which way does each essay lean, and how many occurrences is that lean built
on?



<details><summary>Hint</summary>

`CountVectorizer(vocabulary=['upon', 'whilst', 'while', 'on'])`, then
`.div(essays['n_words'], axis=0) * 1000`. Keep the raw counts in a separate
DataFrame before dividing.

</details>

In [ ]:
style_vec = CountVectorizer(vocabulary=[...])
style_matrix = style_vec.fit_transform(essays['text']).toarray()
style_counts = pd.DataFrame(style_matrix,
                            columns=style_vec.get_feature_names_out())
style = style_counts.div(..., axis=0) * 1000
style['author'] = essays['author']
is_joint = essays['author'] == 'Joint'
joint_numbers = essays.loc[is_joint, 'essay']
print(style_counts[is_joint].set_index(joint_numbers))
style[is_joint].set_index(joint_numbers).round(2)

In [ ]:
style.groupby('author').mean().round(2)

*Your answer:*

## Task 4 · The topic of the three essays

Fit `NMF` with four topics on the tf-idf matrix, as in the lecture. Give
each essay its largest topic, print the ten strongest words of the topic
that holds essays 18 to 20, and print the eight strongest tf-idf words of
each of the three essays.

Why can the topic not decide who wrote more of an essay?



<details><summary>Hint</summary>

`NMF(n_components=4, random_state=0, max_iter=1000)`, then
`weights.argmax(axis=1)`. For one essay's strongest words,
`tfidf[17].toarray().ravel().argsort()[::-1][:8]` indexes `terms` (row 17
is essay 18).

</details>

In [ ]:
nmf = NMF(n_components=4, random_state=0, max_iter=1000)
weights = nmf.fit_transform(tfidf)
essays['topic'] = ...
essays.loc[is_joint, ['essay', 'topic']]

In [ ]:
topic = essays.loc[essays['essay'] == 18, 'topic'].iloc[0]
top = nmf.components_[topic].argsort()[::-1][:10]
print(topic, ', '.join(terms[top]))

In [ ]:
for n in [18, 19, 20]:
    row = tfidf[...].toarray().ravel()
    print(n, ', '.join(terms[row.argsort()[::-1][:8]]))

*Your answer:*